# Semi-Supervised Learning & Deep Clustering

## Imports

In [4]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
from sklearn.cluster import KMeans
import random

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## Data Loading

In [5]:
# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset= torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create labeled and unlabeled subsets
labeled_indices = []
targets = np.array([train_dataset[i][1] for i in range(len(train_dataset))])
for cls in range(10):
    cls_indices = np.where(targets == cls)[0]
    labeled_indices.extend(np.random.choice(cls_indices, 100, replace=False))
unlabeled_indices = list(set(range(len(train_dataset))) - set(labeled_indices))


labeled_dataset = Subset(train_dataset, labeled_indices)
unlabeled_dataset = Subset(train_dataset, unlabeled_indices)

# Data loaders
labeled_loader = DataLoader(labeled_dataset, batch_size=64, shuffle=True)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [6]:
print(f"Number of labeled samples: {len(labeled_dataset)}")
print(f"Number of unlabeled samples: {len(unlabeled_dataset)}")

Number of labeled samples: 1000
Number of unlabeled samples: 59000


## $\Pi$-Model

### Model Definition

**$\Pi$-Model** is a classic architecture in **Semi-Supervised Learning.** It relies in the concept of **consistency regularization:** it assumes that if the same input is given twice with slight stochastic changes, the output should remain roughly the same.

The stochastic permutation is handled by **Dropout:** because Dropout randomly deactivates neurons, the model's internal path is different every time we run a forward pass.

In [7]:
class PiModelCNN(nn.Module):
    def __init__(self):
        super(PiModelCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.dropout1 = nn.Dropout(0.25)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.dropout2 = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64 * 7 * 7, 64)
        self.dropout3 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):

        x = self.pool1(F.relu(self.conv1(x)))
        x = self.dropout1(x)
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.dropout2(x)
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.dropout3(x)
        x = self.fc2(x)
        return torch.softmax(x, dim=1)

### Training Loop

When training a $\Pi$-Model we try at the same time to teach the model how to predict **similar outputs for slightly different inputs** and how to correctly classify the **few labeled samples** we dispose of.

This means that the Loss Function is composed of **two terms:** the first one refers to the **Supervised Loss** (how good the model is at classifying labeled data), and the second one refers to the **Consistency Loss** (how good the model is at producing similar outputs given similar input).

The final loss is given by:

$$Loss = \text{Supervised Loss} + \lambda(t) \cdot \text{Consistency Loss}$$

where **$\lambda$(t)** is used to weight more one loss over the other, preventing one of the two being suffocated.

In [8]:
# Initialize the device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


# Initialize model, loss, and optimizer
pi_model = PiModelCNN().to(device)
criterion_ce = nn.CrossEntropyLoss()
criterion_mse = nn.MSELoss()
optimizer_pi = optim.Adam(pi_model.parameters(), lr=0.001)

In [9]:
def train_pi_model(model, labeled_loader, unlabeled_loader, epochs=300):
    model.train()

    # Define the weight of criterion_mse against criterion_ce
    consistency_weight = 10.0

    for epoch in range(epochs):
        total_loss = 0
        for (labeled_data, labeled_targets), (unlabeled_data, _) in zip(labeled_loader, unlabeled_loader):
            labeled_data    = labeled_data.to(device)
            labeled_targets = labeled_targets.to(device)
            unlabeled_data  = unlabeled_data.to(device)
            
            labeled_preds = model(labeled_data)
            sup_loss = criterion_ce(labeled_preds, labeled_targets)

            # Two forward passes for unlabeled data with different dropout
            model.train()
            unlabeled_preds1 = model(unlabeled_data)
            unlabeled_preds2 = model(unlabeled_data)

            # Compute consistency loss
            cons_loss = criterion_mse(unlabeled_preds1, unlabeled_preds2)

            # Combine losses
            loss = sup_loss + consistency_weight * cons_loss

            # Backward pass
            optimizer_pi.zero_grad()
            loss.backward()
            optimizer_pi.step()

            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f'Epoch {epoch}, Loss: {total_loss / len(labeled_loader):.4f}')

In [10]:
# Train the Π-Model
train_pi_model(pi_model, labeled_loader, unlabeled_loader, epochs=300)

Epoch 0, Loss: 2.3001
Epoch 10, Loss: 1.9897
Epoch 20, Loss: 1.8771
Epoch 30, Loss: 1.8360
Epoch 40, Loss: 1.8030
Epoch 50, Loss: 1.7886
Epoch 60, Loss: 1.7759
Epoch 70, Loss: 1.7466
Epoch 80, Loss: 1.7210
Epoch 90, Loss: 1.6988
Epoch 100, Loss: 1.6872
Epoch 110, Loss: 1.6835
Epoch 120, Loss: 1.6432
Epoch 130, Loss: 1.6399
Epoch 140, Loss: 1.6658
Epoch 150, Loss: 1.6736
Epoch 160, Loss: 1.6371
Epoch 170, Loss: 1.6450
Epoch 180, Loss: 1.6344
Epoch 190, Loss: 1.6455
Epoch 200, Loss: 1.6430
Epoch 210, Loss: 1.5931
Epoch 220, Loss: 1.6094
Epoch 230, Loss: 1.5845
Epoch 240, Loss: 1.6060
Epoch 250, Loss: 1.5906
Epoch 260, Loss: 1.5928
Epoch 270, Loss: 1.5952
Epoch 280, Loss: 1.5941
Epoch 290, Loss: 1.5920


## Deep Clustering

**Deep Clustering** is a model typically used to solve Unsupervised Learning. Here we adapt it for Semi-Supervised Learning.

This model focuses on the **structure **of data: it tries to force unlabeled data into **distinct, tight groups**, based on similarities or dissimilarities between the samples.

### Model Definition

In [11]:
class ClusteringLayer(nn.Module):
    def __init__(self, n_clusters=10):
        super(ClusteringLayer, self).__init__()
        # Number of clusters we have to find
        self.n_clusters = n_clusters

        # Coordinates of the centers of the 10 clusters
        # These are trainable weights in a 10-dimensional space
        self.cluster_centers = nn.Parameter(torch.randn(n_clusters, 10))

    def forward(self, x):
        # Compute the Squared Euclidean Distance between the input data sample x
        # and every cluster center
        d = torch.sum((x.unsqueeze(1) - self.cluster_centers) ** 2, dim=2)

        # Converts a distance into a similarity score between 0 and 1
        # using a Student's t-distribution
        q = 1.0 / (1.0 + d)

        # Sharpening the similarity scores
        q = q ** 2

        # Normalize (ensure scores sum up to 1)
        q = q / q.sum(dim=1, keepdim=True)

        return q

class DeepClusteringModel(nn.Module):
    def __init__(self):
        super(DeepClusteringModel, self).__init__()

        # Define the feature extractor
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

        # Define classifier
        self.classifier = nn.Linear(10, 10)  

        # Define clustering
        self.clustering = ClusteringLayer(n_clusters=10)

    def forward(self, x, mode='both'):
        features = self.encoder(x)
        if mode == 'classifier':
            return torch.softmax(self.classifier(features), dim=1)
        elif mode == 'clustering':
            return self.clustering(features)
        else:
            return torch.softmax(self.classifier(features), dim=1), self.clustering(features)

### Training Loop

Training involves two phases:


1.   **Supervised Pre-training:** we use the labeled data we have to teach the model an **initial representation** of the data. In this way the model should be able to extract relevant features to distinguish between different classes.
We then use these features to **initialize the centers of the clusters.** Instead of assigning random values, we start from a **solid baseline,** using as initial values the center of the clusters of features extracted by the model.
2.   **Actual Training:** in this phase we teach the model how to correctly **classify labeled samples** and, at the same time, how to assign each unlabeled sample to the **correct cluster.**
This last step is done through a technique known as **self-training:** given a unlabeled samples, the model predicts a **soft guess q.** This soft guess is then **sharpened**, producing a target distribution p, which is a more confident prediction. The goal is then to **minimize the difference** between q and p, pushing q toward p.

Since q and p are distributions telling the probability of the given sample to belong to each cluster, the loss between them is computed using the **Kullback-Leibler (KL) Divergence:**

$$KL(P || Q) = \sum P(i) \log \frac{P(i)}{Q(i)}$$

This metric measures the **distance **between two probability distributions or, more precisely, how well a given probability q approximates a target probability p.


The final loss function is given by a **weighted sum** between the classification loss and the KL Divergence.

In [12]:
# Initialize model and optimizer
dc_model = DeepClusteringModel().to(device)
optimizer_dc = optim.Adam(dc_model.parameters(), lr=0.001)

In [13]:
# This function exploits the few labeled data we have to pre-train the model
def pretrain_dc_model(model, labeled_loader, epochs=30):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for data, targets in labeled_loader:
            data, targets = data.to(device), targets.to(device)

            preds = model(data, mode='classifier')
            loss = criterion_ce(preds, targets)

            optimizer_dc.zero_grad()
            loss.backward()
            optimizer_dc.step()
            
            total_loss += loss.item()
        print(f'Pretrain Epoch {epoch}, Loss: {total_loss / len(labeled_loader):.4f}')

# This function takes the features extracted by the model, uses
# KMeans to find the center of each cluster and initialize the
# clusters of the model using the center of the clusters found by KMeans
def initialize_clusters(model, labeled_loader):
    model.eval()
    features = []
    with torch.no_grad():
        for data, _ in labeled_loader:
            data = data.to(device)

            feat = model.encoder(data)
            features.append(feat.cpu().numpy())

    features = np.concatenate(features, axis=0)

    kmeans = KMeans(n_clusters=10, random_state=42)

    # Fit KMeans model
    kmeans.fit(features)

    model.clustering.cluster_centers.data = torch.tensor(kmeans.cluster_centers_).to(device)

def train_dc_model(model, labeled_loader, unlabeled_loader, epochs=300):
    model.train()
    clustering_weight = 0.1
    criterion_kl = nn.KLDivLoss(reduction='batchmean')

    for epoch in range(epochs):
        total_loss = 0
        for (labeled_data, labeled_targets), (unlabeled_data, _) in zip(labeled_loader, unlabeled_loader):
            labeled_data, labeled_targets = labeled_data.to(device), labeled_targets.to(device)
            unlabeled_data = unlabeled_data.to(device)

            # Forward pass for labeled data
            sup_loss = criterion_ce(model(labeled_data, mode='classifier'), labeled_targets)

            # Forward pass for unlabeled data
            _, cluster_probs = model(unlabeled_data, mode='both')

            # Compute target distribution (sharpened)
            p = cluster_probs ** 2 / torch.sum(cluster_probs, dim=0)
            p = p / torch.sum(p, dim=1, keepdim=True)


            cluster_loss = criterion_kl(torch.log(cluster_probs), p)

            # Combine losses
            loss = sup_loss + clustering_weight * cluster_loss

            # Backward pass
            optimizer_dc.zero_grad()
            loss.backward()
            optimizer_dc.step()

            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f'Epoch {epoch}, Loss: {total_loss / len(labeled_loader):.4f}')

In [14]:
pretrain_dc_model(dc_model, labeled_loader, epochs=30)
initialize_clusters(dc_model, labeled_loader)
train_dc_model(dc_model, labeled_loader, unlabeled_loader, epochs=300)

Pretrain Epoch 0, Loss: 2.2692
Pretrain Epoch 1, Loss: 2.0424
Pretrain Epoch 2, Loss: 1.8369
Pretrain Epoch 3, Loss: 1.7599
Pretrain Epoch 4, Loss: 1.7194
Pretrain Epoch 5, Loss: 1.6890
Pretrain Epoch 6, Loss: 1.6755
Pretrain Epoch 7, Loss: 1.6474
Pretrain Epoch 8, Loss: 1.6308
Pretrain Epoch 9, Loss: 1.6205
Pretrain Epoch 10, Loss: 1.6152
Pretrain Epoch 11, Loss: 1.6049
Pretrain Epoch 12, Loss: 1.6014
Pretrain Epoch 13, Loss: 1.5948
Pretrain Epoch 14, Loss: 1.5935
Pretrain Epoch 15, Loss: 1.5949
Pretrain Epoch 16, Loss: 1.5906
Pretrain Epoch 17, Loss: 1.5823
Pretrain Epoch 18, Loss: 1.5770
Pretrain Epoch 19, Loss: 1.5740
Pretrain Epoch 20, Loss: 1.5769
Pretrain Epoch 21, Loss: 1.5717
Pretrain Epoch 22, Loss: 1.5503
Pretrain Epoch 23, Loss: 1.5118
Pretrain Epoch 24, Loss: 1.5025
Pretrain Epoch 25, Loss: 1.4828
Pretrain Epoch 26, Loss: 1.4786
Pretrain Epoch 27, Loss: 1.4766
Pretrain Epoch 28, Loss: 1.4752
Pretrain Epoch 29, Loss: 1.4729
Epoch 0, Loss: 1.4835
Epoch 10, Loss: 1.4796
Epoch

## Evaluation and Model Comparison

In [20]:
def evaluate_model(model, test_loader, mode='classifier'):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, targets in test_loader:
            data, targets = data.to(device), targets.to(device)
            # Check if model is DeepClusteringModel (supports mode) or PiModelCNN (does not)
            if isinstance(model, DeepClusteringModel):
                preds = model(data, mode=mode)
            else:  # PiModelCNN
                preds = model(data)
            _, predicted = torch.max(preds, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    return 100 * correct / total

# Train supervised baseline
sup_model = PiModelCNN().to(device)
optimizer_sup = optim.Adam(sup_model.parameters(), lr=0.001)

def train_sup_model(model, labeled_loader, epochs=300):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for data, targets in labeled_loader:
            data, targets = data.to(device), targets.to(device)
            preds = model(data)
            loss = criterion_ce(preds, targets)
            optimizer_sup.zero_grad()
            loss.backward()
            optimizer_sup.step()
            total_loss += loss.item()
        if epoch % 10 == 0:
            print(f'Supervised Epoch {epoch}, Loss: {total_loss / len(labeled_loader):.4f}')

train_sup_model(sup_model, labeled_loader)

Supervised Epoch 0, Loss: 2.2831
Supervised Epoch 10, Loss: 1.6869
Supervised Epoch 20, Loss: 1.5735
Supervised Epoch 30, Loss: 1.5516
Supervised Epoch 40, Loss: 1.5362
Supervised Epoch 50, Loss: 1.5061
Supervised Epoch 60, Loss: 1.5114
Supervised Epoch 70, Loss: 1.5089
Supervised Epoch 80, Loss: 1.5028
Supervised Epoch 90, Loss: 1.4957
Supervised Epoch 100, Loss: 1.4916
Supervised Epoch 110, Loss: 1.4887
Supervised Epoch 120, Loss: 1.4906
Supervised Epoch 130, Loss: 1.4851
Supervised Epoch 140, Loss: 1.4839
Supervised Epoch 150, Loss: 1.4856
Supervised Epoch 160, Loss: 1.4855
Supervised Epoch 170, Loss: 1.4978
Supervised Epoch 180, Loss: 1.4839
Supervised Epoch 190, Loss: 1.4871
Supervised Epoch 200, Loss: 1.4826
Supervised Epoch 210, Loss: 1.4789
Supervised Epoch 220, Loss: 1.4821
Supervised Epoch 230, Loss: 1.4830
Supervised Epoch 240, Loss: 1.4756
Supervised Epoch 250, Loss: 1.4834
Supervised Epoch 260, Loss: 1.4764
Supervised Epoch 270, Loss: 1.4791
Supervised Epoch 280, Loss: 1.4

In [21]:
# Evaluate all models
pi_accuracy = evaluate_model(pi_model, test_loader)
dc_accuracy = evaluate_model(dc_model, test_loader)
sup_accuracy = evaluate_model(sup_model, test_loader)


print(f'Π-Model Test Accuracy: {pi_accuracy:.2f}%')
print(f'Deep Clustering Test Accuracy: {dc_accuracy:.2f}%')
print(f'Supervised Baseline Test Accuracy: {sup_accuracy:.2f}%')

Π-Model Test Accuracy: 97.68%
Deep Clustering Test Accuracy: 97.47%
Supervised Baseline Test Accuracy: 96.74%
